In [1]:
# ==========================================
# Step 1: Environment Setup & Dataset Ingestion
# ==========================================

# 1. ติดตั้งไลบรารีเสริม
!pip install -q kagglehub scikit-learn seaborn matplotlib torchsummary

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import kagglehub

# 2. ล็อกค่า Random Seed เพื่อให้ผลลัพธ์การทดลองทำซ้ำได้ (Reproducible)
SEED = 101
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. ดาวน์โหลดชุดข้อมูล Cars Body Type Cropped จาก Kaggle
dataset_path = kagglehub.dataset_download("ademboukhris/cars-body-type-cropped")
print(f"Dataset downloaded to: {dataset_path}")

Using device: cuda


100%|██████████| 1.34G/1.34G [00:19<00:00, 75.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ademboukhris/cars-body-type-cropped/versions/1


In [3]:
# ==========================================
# Step 2: Dataset Exploration & Data Splitting (70/15/15)
# ==========================================

# 1. ตรวจสอบโฟลเดอร์ย่อยของชุดข้อมูล
subdirs = [os.path.join(dataset_path, d) for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
data_dir = subdirs[0] if len(subdirs) == 1 else dataset_path

# 2. อ่านข้อมูลภาพและนับจำนวนแต่ละคลาส
raw_dataset = datasets.ImageFolder(root=data_dir)
class_names = raw_dataset.classes
num_classes = len(class_names)

print(f"Total images found: {len(raw_dataset):,} images")
print(f"Number of classes ({num_classes}): {class_names}")

# 3. แบ่งข้อมูลแบบปลอด Data Leakage ด้วย Seed 101
total_len = len(raw_dataset)
indices = list(range(total_len))
random.shuffle(indices)

train_split = int(0.70 * total_len)
val_split = int(0.85 * total_len)

train_idx = indices[:train_split]
val_idx = indices[train_split:val_split]
test_idx = indices[val_split:]

print(f"Data Split Summary:")
print(f"- Train set:      {len(train_idx):,} samples ({len(train_idx)/total_len*100:.1f}%)")
print(f"- Validation set: {len(val_idx):,} samples ({len(val_idx)/total_len*100:.1f}%)")
print(f"- Test set:       {len(test_idx):,} samples ({len(test_idx)/total_len*100:.1f}%)")

Total images found: 7,549 images
Number of classes (3): ['test', 'train', 'valid']
Data Split Summary:
- Train set:      5,284 samples (70.0%)
- Validation set: 1,132 samples (15.0%)
- Test set:       1,133 samples (15.0%)
